# Advanced 3 — SHAP-пояснення

Глобальні та локальні пояснення через **SHAP** (TreeExplainer на XGBoost). ⚠️ Рахуємо на **train** (де модель має сигнал): на дрейфованому тесті AUC≈0.5, тож тестові важливості — шум. Графіки зберігаються у `reports/plots/`.

> Залежності: `%run 03_data_prep.ipynb`.

In [ ]:
%run 03_data_prep.ipynb

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import numpy as np, shap, xgboost as xgb
from pathlib import Path
PLOTS = Path('reports/plots'); PLOTS.mkdir(parents=True, exist_ok=True)
Xtr, ytr, Xte, yte = b['X_train'], b['y_train'], b['X_test'], b['y_test']
model = xgb.XGBClassifier(max_depth=3, n_estimators=200, learning_rate=0.05,
                          tree_method='hist', random_state=42).fit(Xtr, ytr)
explainer = shap.TreeExplainer(model)
sv = explainer(Xtr)            # SHAP на TRAIN
print('SHAP values shape:', sv.values.shape)

### Глобально: які фічі найважливіші (beeswarm + bar)

In [ ]:
plt.figure(); shap.plots.beeswarm(sv, max_display=12, show=False)
plt.tight_layout(); plt.savefig(PLOTS/'shap_beeswarm.png', dpi=120, bbox_inches='tight'); plt.show()
import numpy as np
imp = np.abs(sv.values).mean(0)
order = np.argsort(imp)[::-1][:10]
print('Топ-10 за |SHAP| (на train):')
for i in order: print(f'  {features.FEATURE_COLUMNS[i]:18s} {imp[i]:.4f}')

### Локально: пояснення одного прогнозу (waterfall)

In [ ]:
i = 0  # ← індекс прикладу
plt.figure(); shap.plots.waterfall(sv[i], max_display=10, show=False)
plt.tight_layout(); plt.savefig(PLOTS/'shap_waterfall.png', dpi=120, bbox_inches='tight'); plt.show()
print('Пояснення для прикладу', i, '— топ-внески:')
ssingle = sorted(zip(features.FEATURE_COLUMNS, sv.values[i]), key=lambda t: abs(t[1]), reverse=True)[:5]
for name, val in ssingle:
    print(f"  {('+' if val>0 else '')}{val:.3f}  {name}")

### Висновок

SHAP узгоджується з permutation-важливостями проєкту (тривалість-бакети, час доби, emoji, sentiment) — слабкі, але інтерпретовані ефекти. Для деплоєної **лінійної** моделі точні signed-внески дає сам `inference` (`coef × scaled`), тож SHAP тут — переважно для **нелінійного** XGB-контексту. Не переоцінюй: ефекти малі, бо сигнал слабкий.